In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import random
import numpy as np
import re
import os
from collections import Counter
import math

# Control randomness (roughly)
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Device configuration (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
learning_rate = 0.008
batch_size = 68
num_epochs = 28
embedding_dim = 32
num_heads = 8
num_layers = 1
dim_feedforward = 32
max_length = 200  # Maximum sequence length
vocab_size = 10000  # Maximum vocabulary size

# ==================== Data Preparation ====================
# Download AG News dataset (if not already downloaded)
print("Checking for AG News dataset...")
train_file = 'train.csv'
test_file = 'test.csv'

if os.path.exists(train_file) and os.path.exists(test_file):
    print("Dataset already exists. Skipping download.")
else:
    print("Downloading AG News dataset...")
    if not os.path.exists(train_file):
        !wget -q https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv
    if not os.path.exists(test_file):
        !wget -q https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv
    print("Download complete.")

# Load AG News data from CSV files
def load_ag_news_data(filename):
    texts = []
    labels = []

    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            # CSV format: "class","title","description"
            parts = line.strip().split('","')
            if len(parts) >= 3:
                # Extract label (subtract 1 to make it 0-indexed)
                label = int(parts[0].replace('"', '')) - 1
                # Combine title and description
                title = parts[1]
                description = parts[2].replace('"', '')
                text = title + ' ' + description

                texts.append(text)
                labels.append(label)

    return texts, labels

train_texts, train_labels = load_ag_news_data('train.csv')
test_texts, test_labels = load_ag_news_data('test.csv')

# Simple tokenizer
def simple_tokenizer(text):
    # Convert to lowercase and split by non-alphanumeric characters
    text = text.lower()  # Convert text to lowercase
    tokens = re.findall(r'\b\w+\b', text)  # e.g., "Hello World!" -> ['hello', 'world']
    return tokens

# Build vocabulary
word_counts = Counter()  # Create a counter to store word frequencies
for text in train_texts:
    tokens = simple_tokenizer(text)  # e.g., "Hi hi bye" -> ['hi', 'hi', 'bye']
    word_counts.update(tokens)  # e.g., Update counts: {'hi': 2, 'bye': 1}

# Keep most common words in vocabulary
most_common = word_counts.most_common(vocab_size - 3)  # Reserve space for <pad>, <unk>, and <cls>
vocab = {'<pad>': 0, '<unk>': 1, '<cls>': 2}
for idx, (word, _) in enumerate(most_common, start=3):
    vocab[word] = idx
    # e.g., {'<pad>': 0, '<unk>': 1, '<cls>': 2, 'the': 3, 'a': 4, 'news': 5, ...}

print(f"Vocabulary size: {len(vocab)}")

# Text processing pipeline
def text_to_indices(text, vocab, max_length):
    tokens = simple_tokenizer(text)
    # e.g., "A xyz news" -> ['a', 'xyz', 'news']
    indices = [vocab['<cls>']] + [vocab.get(token, vocab['<unk>']) for token in tokens]
    # e.g., ['a', 'xyz', 'news'] -> [2, 3, 1, 4] ('xyz' is unknown)

    # Pad or truncate to max_length
    if len(indices) < max_length:
        indices = indices + [vocab['<pad>']] * (max_length - len(indices))
        # e.g., [2, 3, 1, 4] (max_length=5) -> [2, 3, 1, 4, 0]
    else:
        indices = indices[:max_length]
        # e.g., [2, 3, 1, 4, 5, 6, 7] (max_length=5) -> [2, 3, 1, 4, 5]

    return torch.tensor(indices, dtype=torch.long)

# Custom Dataset class
class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length):
        self.data = []
        self.labels = []

        for text, label in zip(texts, labels):
            self.data.append(text_to_indices(text, vocab, max_length))
            self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Create datasets
train_dataset_full = AGNewsDataset(train_texts, train_labels, vocab, max_length)
test_dataset_full = AGNewsDataset(test_texts, test_labels, vocab, max_length)

# Use a subset
g = torch.Generator()
g.manual_seed(seed)
train_subset_indices = torch.randperm(len(train_dataset_full), generator=g)[:10000]
test_subset_indices = torch.randperm(len(test_dataset_full), generator=g)[:2000]
train_dataset = Subset(train_dataset_full, train_subset_indices)
test_dataset = Subset(test_dataset_full, test_subset_indices)

# Data loader
train_loader = DataLoader(dataset=train_dataset,
                         batch_size=batch_size,
                         shuffle=True)
test_loader = DataLoader(dataset=test_dataset,
                        batch_size=batch_size,
                        shuffle=False)

# ==================== Model Definition ====================

class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_length=5000):
        super(PositionalEncoding, self).__init__()
        # Create positional encoding matrix
        # PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
        # PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

        pe = torch.zeros(max_length, embedding_dim)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        # e.g., position: [[0], [1], [2], ..., [max_length-1]]

        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * (-math.log(10000.0) / embedding_dim))
        # e.g., div_term computes: 10000^(-2i/d_model) for i in [0, 1, 2, ...]

        pe[:, 0::2] = torch.sin(position * div_term)  # Even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd indices

        pe = pe.unsqueeze(0)  # Add batch dimension: (1, max_length, embedding_dim)
        self.register_buffer('pe', pe)  # Register as buffer (not a parameter)

    def forward(self, x):
        # x shape: (batch_size, seq_length, embedding_dim)
        # Add positional encoding to input embeddings
        x = x + self.pe[:, :x.size(1), :] # pe is broadcast across the batch dimension of x
        return x

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, dim_feedforward, num_classes, max_length):
        super(TransformerClassifier, self).__init__()

        # Embedding layer: Maps word indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # e.g., index 4 ('news') -> [0.1, 0.5, -0.2, ...] (embedding_dim dimensions)

        # Positional encoding: Adds position information to embeddings
        self.pos_encoder = PositionalEncoding(embedding_dim, max_length)

        # Transformer encoder layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            batch_first=True  # Input/output shape is (batch_size, seq_length, features)
        )

        # Stack multiple transformer encoder layers
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Fully connected layer for classification
        self.fc = nn.Linear(embedding_dim, num_classes)

        self.embedding_dim = embedding_dim

    def forward(self, x):
        # x shape: (batch_size, seq_length)

        # Create padding mask (True for padding tokens)
        padding_mask = (x == 0)  # (batch_size, seq_length)

        # Embed tokens
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)
        # e.g., [[4, 50, 2]] -> [[[0.1,...], [0.8,...], [0.2,...]]]

        # Add positional encoding
        embedded = self.pos_encoder(embedded)  # (batch_size, seq_length, embedding_dim)

        # Pass through transformer encoder
        transformer_out = self.transformer_encoder(embedded, src_key_padding_mask=padding_mask)
        # transformer_out shape: (batch_size, seq_length, embedding_dim)

        # Alternative: Use [CLS] token or take first token
        cls_output = transformer_out[:, 0, :]  # (batch_size, embedding_dim)

        # Classification
        out = self.fc(cls_output)  # (batch_size, num_classes)
        return out

# Create model instance
model = TransformerClassifier(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    num_classes=4,  # AG News has 4 classes
    max_length=max_length,
).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

# ==================== Training the model ====================

model.train()  # Training mode
total_step = len(train_loader)
for epoch in range(num_epochs):
    sum_epoch_loss = 0.0
    for i, (texts, labels) in enumerate(train_loader):
        texts, labels = texts.to(device), labels.to(device)

        # Forward pass
        outputs = model(texts)
        loss = criterion(outputs, labels)
        sum_epoch_loss += loss.item()

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Print epoch average loss
    avg_epoch_loss = sum_epoch_loss / total_step
    print(f"Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_epoch_loss:.4f}")

# ==================== Testing the model ====================

model.eval()  # Testing mode
with torch.no_grad():
    correct = 0
    total = 0
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)

        # Get predicted label
        outputs = model(texts)
        predicted = outputs.argmax(dim=1)

        total += texts.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the {len(test_dataset)} test samples: {100 * correct / total:.2f} %')

Using device: cuda
Checking for AG News dataset...
Dataset already exists. Skipping download.
Vocabulary size: 10000

Model architecture:
TransformerClassifier(
  (embedding): Embedding(10000, 32, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=32, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=32, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=32, out_features=4, 